<h1 align="center">HW3 · KWS</h1>

In [ ]:
!pip -q install librosa soundfile scikit-learn tqdm
!pip -q install pandas==2.2.2

!apt-get -qq update
!apt-get -qq install -y ffmpeg


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cpu"
print("device:", DEVICE)

try:
    torchaudio.set_audio_backend("ffmpeg")
    print("torchaudio backend:", torchaudio.get_audio_backend())
except Exception as e:
    print("backend not set:", e)


device: cpu
torchaudio backend: None


/tmp/ipython-input-371905528.py:29: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("ffmpeg")
/tmp/ipython-input-371905528.py:30: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  print("torchaudio backend:", torchaudio.get_audio_backend())


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_TRAIN = Path("/content/drive/MyDrive/train_data.tar")
DRIVE_TEST  = Path("/content/drive/MyDrive/test_data.tar")
assert DRIVE_TRAIN.exists(), DRIVE_TRAIN
assert DRIVE_TEST.exists(), DRIVE_TEST

LOCAL_TRAIN = Path("/content/train_data.tar")
LOCAL_TEST  = Path("/content/test_data.tar")

def cp_if_needed(src: Path, dst: Path):
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return
    subprocess.run(["cp", str(src), str(dst)], check=True)

cp_if_needed(DRIVE_TRAIN, LOCAL_TRAIN)
cp_if_needed(DRIVE_TEST,  LOCAL_TEST)

!ls -lh /content/train_data.tar /content/test_data.tar

WORK = Path("/content/data_work")
TRAIN_RAW = WORK / "train_raw"
TEST_RAW  = WORK / "test_raw"
TRAIN_RAW.mkdir(parents=True, exist_ok=True)
TEST_RAW.mkdir(parents=True, exist_ok=True)

def extract_if_needed(tar_path: Path, dst: Path):
    marker = dst / ".extracted_ok"
    if marker.exists():
        return
    subprocess.run(["tar", "-xf", str(tar_path), "-C", str(dst)], check=True)
    marker.touch()

extract_if_needed(LOCAL_TRAIN, TRAIN_RAW)
extract_if_needed(LOCAL_TEST,  TEST_RAW)

print("TRAIN_RAW:", TRAIN_RAW)
print("TEST_RAW :", TEST_RAW)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 707M Dec 18 21:59 /content/test_data.tar
-rw------- 1 root root 2.4G Dec 18 21:52 /content/train_data.tar
TRAIN_RAW: /content/data_work/train_raw
TEST_RAW : /content/data_work/test_raw


In [ ]:
TRAIN_AUDIO = Path("/content/data_work/train_raw/train_opus/audio")
TEST_AUDIO  = Path("/content/data_work/test_raw/test_opus/audio")
BOUNDS_PATH = Path("/content/data_work/train_raw/train_opus/word_bounds.json")

assert TRAIN_AUDIO.exists(), TRAIN_AUDIO
assert TEST_AUDIO.exists(), TEST_AUDIO
assert BOUNDS_PATH.exists(), BOUNDS_PATH

print("TRAIN_AUDIO:", TRAIN_AUDIO)
print("TEST_AUDIO :", TEST_AUDIO)
print("BOUNDS     :", BOUNDS_PATH)


TRAIN_AUDIO: /content/data_work/train_raw/train_opus/audio
TEST_AUDIO : /content/data_work/test_raw/test_opus/audio
BOUNDS     : /content/data_work/train_raw/train_opus/word_bounds.json


### Списки файлов и разметка
---

In [ ]:
with open(BOUNDS_PATH, "r", encoding="utf-8") as f:
    bounds = json.load(f)

train_files = sorted([p for p in TRAIN_AUDIO.glob("*.opus") if not p.name.startswith("._")])
test_files  = sorted([p for p in TEST_AUDIO.glob("*.opus")  if not p.name.startswith("._")])

train_ids = [p.stem for p in train_files]
test_ids  = [p.stem for p in test_files]

labels = {tid: (1 if tid in bounds else 0) for tid in train_ids}

num_pos = sum(labels.values())
num_neg = len(labels) - num_pos
print("Train:", len(train_ids), "pos:", num_pos, "neg:", num_neg)
print("Test :", len(test_ids))


Train: 90000 pos: 45000 neg: 45000
Test : 27000


### Сплит
---

In [ ]:
y_all = np.array([labels[i] for i in train_ids], dtype=int)

train_ids_split, val_ids_split = train_test_split(
    train_ids, test_size=0.15, random_state=SEED, stratify=y_all
)

print("train ids:", len(train_ids_split), "val ids:", len(val_ids_split))


train ids: 76500 val ids: 13500


### Датасет
---

In [ ]:
SR = 16000
WIN_SEC = 2.5
WIN_SAMPLES = int(SR * WIN_SEC)

def load_audio_1d(path: Path, target_sr=SR) -> torch.Tensor:
    wav, sr = torchaudio.load(str(path))  # (C,T)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    wav = wav[0]
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    return wav  # [T]

def crop_or_pad(wav: torch.Tensor, start: int, length: int) -> torch.Tensor:
    T = wav.numel()
    if T <= 0:
        return torch.zeros(length)
    start = max(0, min(int(start), max(0, T - 1)))
    end = start + length
    chunk = wav[start:min(end, T)]
    if chunk.numel() < length:
        chunk = torch.nn.functional.pad(chunk, (0, length - chunk.numel()))
    return chunk

class PhraseWindowDataset(Dataset):
    def __init__(self, audio_dir: Path, ids, bounds_dict, labels_dict, train_mode=True):
        self.audio_dir = audio_dir
        self.ids = list(ids)
        self.bounds = bounds_dict
        self.labels = labels_dict
        self.train_mode = train_mode

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        tid = self.ids[idx]
        path = self.audio_dir / f"{tid}.opus"
        y = float(self.labels[tid])

        wav = load_audio_1d(path, SR)
        T = wav.numel()

        if self.train_mode:
            if y == 1.0:
                st, en = self.bounds[tid]
                st_s = int(max(0.0, st) * SR)
                en_s = int(min(float(T) / SR, en) * SR)
                center = (st_s + en_s) // 2

                jitter = int(0.2 * WIN_SAMPLES)
                start = center - WIN_SAMPLES // 2 + random.randint(-jitter, jitter)
                start = max(0, min(start, max(0, T - WIN_SAMPLES)))
            else:
                start = 0 if T <= WIN_SAMPLES else random.randint(0, T - WIN_SAMPLES)
        else:
            start = 0 if T <= WIN_SAMPLES else (T - WIN_SAMPLES) // 2

        chunk = crop_or_pad(wav, start, WIN_SAMPLES)
        return chunk, torch.tensor(y, dtype=torch.float32), tid

def collate_fn(batch):
    waves = torch.stack([b[0] for b in batch], dim=0)
    y = torch.stack([b[1] for b in batch], dim=0)
    ids = [b[2] for b in batch]
    return waves, y, ids


### Log-mel и CNN модель
---

In [ ]:
class LogMel(nn.Module):
    def __init__(self, sr=SR, n_fft=400, hop=160, n_mels=64, f_min=50, f_max=7600):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop,
            n_mels=n_mels, f_min=f_min, f_max=f_max, power=2.0
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

    def forward(self, waves):
        x = self.mel(waves)
        x = self.db(x)
        x = (x - x.mean(dim=(1,2), keepdim=True)) / (x.std(dim=(1,2), keepdim=True) + 1e-6)
        return x

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.head(x).squeeze(-1)

fe = LogMel().to(DEVICE)
model = SimpleCNN().to(DEVICE)
print("params:", sum(p.numel() for p in model.parameters()))


params: 27745


### DataLoader
---

In [ ]:
train_ds = PhraseWindowDataset(TRAIN_AUDIO, train_ids_split, bounds, labels, train_mode=True)
val_ds   = PhraseWindowDataset(TRAIN_AUDIO, val_ids_split,   bounds, labels, train_mode=False)

BATCH = 16

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2,
                          pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2,
                          pin_memory=True, collate_fn=collate_fn)

len(train_ds), len(val_ds)


(76500, 13500)

### Метрики
---

In [ ]:
def score_from_counts(tp, fp, tn, fn):
    num_pos = tp + fn
    num_neg = tn + fp
    frr = (fn / num_pos) if num_pos > 0 else 0.0
    far = (fp / num_neg) if num_neg > 0 else 0.0
    a = 1.0 - frr
    b = 1.0 - far
    score = 0.0 if (a + b) == 0 else (2 * a * b / (a + b))
    return score, frr, far

@torch.no_grad()
def eval_on_val_windows(model, loader):
    model.eval()
    probs, ys = [], []
    for waves, y, _ in loader:
        waves = waves.to(DEVICE, non_blocking=True)
        y = y.cpu().numpy().astype(int)

        x = fe(waves).unsqueeze(1)
        p = torch.sigmoid(model(x)).cpu().numpy()

        probs.append(p)
        ys.append(y)
    return np.concatenate(ys), np.concatenate(probs)

def find_best_threshold(y_true, prob):
    best = (-1.0, 0.5, None)
    for t in np.linspace(0.05, 0.95, 19):
        pred = (prob >= t).astype(int)
        tp = int(((pred==1) & (y_true==1)).sum())
        fp = int(((pred==1) & (y_true==0)).sum())
        tn = int(((pred==0) & (y_true==0)).sum())
        fn = int(((pred==0) & (y_true==1)).sum())
        s, frr, far = score_from_counts(tp, fp, tn, fn)
        if s > best[0]:
            best = (float(s), float(t), (float(frr), float(far)))
    return best


### Обучение
---

In [ ]:
pos_cnt = sum(labels[i] for i in train_ids_split)
neg_cnt = len(train_ids_split) - pos_cnt
pos_weight = torch.tensor([neg_cnt / max(1, pos_cnt)], device=DEVICE, dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 8
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

best_val = -1.0
best_state = None
best_thr = 0.5

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    n = 0

    for waves, y, _ in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        waves = waves.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            x = fe(waves).unsqueeze(1)
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total += float(loss.detach().cpu()) * waves.size(0)
        n += waves.size(0)

    train_loss = total / max(1, n)

    y_true, prob = eval_on_val_windows(model, val_loader)
    val_score, thr, (frr, far) = find_best_threshold(y_true, prob)

    print(f"epoch {epoch}: loss={train_loss:.4f} | val_score={val_score:.4f} thr={thr:.2f} FRR={frr:.3f} FAR={far:.3f}")

    if val_score > best_val:
        best_val = val_score
        best_thr = thr
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print("BEST:", best_val, "thr:", best_thr)
model.load_state_dict(best_state)
model.to(DEVICE).eval()


epoch 1/8:   0%|          | 0/4782 [00:00<?, ?it/s]

epoch 1: loss=0.6239 | val_score=0.6796 thr=0.35 FRR=0.337 FAR=0.303


epoch 2/8:   0%|          | 0/4782 [00:00<?, ?it/s]

epoch 2: loss=0.5568 | val_score=0.7157 thr=0.80 FRR=0.239 FAR=0.325


epoch 3/8:   0%|          | 0/4782 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789a1508e5c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x789a1508e5c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 14

epoch 7: loss=0.4676 | val_score=0.7722 thr=0.70 FRR=0.191 FAR=0.261


epoch 8/8:   0%|          | 0/4782 [00:00<?, ?it/s]

epoch 8: loss=0.4591 | val_score=0.7703 thr=0.40 FRR=0.224 FAR=0.236
BEST: 0.7721710950851292 thr: 0.7


SimpleCNN(
  (conv): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
  )
  (pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (head): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

### Инференс
---

In [ ]:
@torch.no_grad()
def file_max_prob(path: Path, win_samples=WIN_SAMPLES, hop_sec=1.0, batch_windows=64, max_windows=250):
    wav = load_audio_1d(path, SR)
    T = wav.numel()
    if T <= win_samples:
        chunk = crop_or_pad(wav, 0, win_samples).unsqueeze(0).to(DEVICE)
        x = fe(chunk).unsqueeze(1)
        return float(torch.sigmoid(model(x)).item())

    hop = int(hop_sec * SR)
    starts = list(range(0, T - win_samples + 1, hop))
    if starts and starts[-1] != T - win_samples:
        starts.append(T - win_samples)

    if len(starts) > max_windows:
        idx = np.linspace(0, len(starts)-1, max_windows).astype(int)
        starts = [starts[i] for i in idx]

    best = 0.0
    batch = []

    for s in starts:
        batch.append(crop_or_pad(wav, s, win_samples))
        if len(batch) == batch_windows:
            waves = torch.stack(batch, dim=0).to(DEVICE)
            x = fe(waves).unsqueeze(1)
            p = torch.sigmoid(model(x)).detach().cpu().numpy()
            best = max(best, float(p.max()))
            batch = []
            if best >= 0.999:
                return best

    if batch:
        waves = torch.stack(batch, dim=0).to(DEVICE)
        x = fe(waves).unsqueeze(1)
        p = torch.sigmoid(model(x)).detach().cpu().numpy()
        best = max(best, float(p.max()))

    return best

test_probs = []
for p in tqdm(test_files, desc="predict test"):
    test_probs.append(file_max_prob(p, hop_sec=1.0, batch_windows=64, max_windows=250))

test_pred = [1 if pr >= best_thr else 0 for pr in test_probs]
sum(test_pred), len(test_pred)


predict test:   0%|          | 0/27000 [00:00<?, ?it/s]

(20586, 27000)

In [ ]:
sub = pd.DataFrame({"id": test_ids, "label": test_pred})
out_path = Path("/content/submition.csv")
sub.to_csv(out_path, index=False)

print("saved:", out_path, "rows:", len(sub))
sub.head()


saved: /content/submition.csv rows: 27000


,id,label
0,0000219778122723066859323624505982384475,0
1,0000920560142346477464477964040846645823,1
2,0002106775361063830068199242310438122126,1
3,0002161736146841817059430282255903999813,0
4,0002303832386140303186933286284938192307,0
